In [1]:
import numpy as np
import pandas as pd

In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

In [5]:
from gen_ai_hub.proxy.langchain.google_vertexai import GenerativeModel

In [6]:
client = GenerativeModel(model="gemini-2.0-flash")

In [ ]:
client.generate_content(
    contents='Who is president of India',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": list[PersonName],
    },)



TypeError: Message must be initialized with a dict: google.cloud.aiplatform.v1beta1.GenerationConfig

In [7]:
epic = """EPIC: User Authentication System
EPIC Key: AUTH-EPIC-01
Summary: Implement a secure and user-friendly authentication system for the web application.

Description:
This EPIC includes all the work required to create a basic user authentication system. This includes user registration, login, logout, password reset, and session management. The goal is to ensure only authorized users can access protected areas of the application while maintaining a smooth user experience.

Goals:

Enable users to create an account

Allow users to log in and log out securely

Implement password recovery/reset flow

Ensure proper session handling and security practices

Components:

Frontend (React)

Backend (Node.js / Express)

Database (PostgreSQL)

Labels:
authentication, backend, frontend, security

Child Stories/Tasks (examples):

AUTH-01: Create user registration form (Frontend)

AUTH-02: Implement user registration API (Backend)

AUTH-03: Build login/logout UI (Frontend)

AUTH-04: Create login/logout API endpoints (Backend)

AUTH-05: Setup password reset flow (email + frontend + backend)

AUTH-06: Store passwords securely using hashing

AUTH-07: Implement session/token-based authentication

AUTH-08: Add unit and integration tests for auth module

Acceptance Criteria:

Users can register with email and password

Users can log in and out successfully

Passwords are never stored in plaintext

Password reset email is sent with a secure token

Sessions or tokens expire appropriately

Priority: High
Target Release: v1.0"""

In [8]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional, List

In [12]:
class JiraEpic(BaseModel): 
    actors: str = Optional[Field(description='person or system that interacts with the product or feature described in the story')]
    Components: List[str] = Optional[Field(description='a subsection or part of a project that helps you organize and group')]
    
    #Processes/ workflows​
    #Dependencies​
    #Data exchange touch points

class PersonName(BaseModel):
    name: Optional[str] = Field(description='Write down the name of the person in the text')
"""    name: str
    age: int = 35
    gender: Optional[str] = None
    email: Optional[EmailStr] = None
    cgpa: float = Field(gt=0,lt=10, default=5, description='A decimal value representing the cgpa of the student')"""

"    name: str\n    age: int = 35\n    gender: Optional[str] = None\n    email: Optional[EmailStr] = None\n    cgpa: float = Field(gt=0,lt=10, default=5, description='A decimal value representing the cgpa of the student')"

In [27]:
from langchain_core.prompts import PromptTemplate

# template
template = PromptTemplate(
    template="""
You are a technical architect and Jira expert.
Find the Actors and Components in the Jira Epic Story described below:
"{jira_epic}"
""",
input_variables=['jira_epic']
)

In [28]:
prompt = template.invoke({'jira_epic':epic})

In [43]:
proxy_client=get_proxy_client('gen-ai-hub')
chat_llm = ChatOpenAI(proxy_model_name = 'gpt-4o', proxy_client = proxy_client, temperature=0.0)
structured_model = chat_llm.with_structured_output(JiraEpic)
structured_model1 = chat_llm.with_structured_output(PersonName)
#response = structured_model.invoke(epic)
response = structured_model1.invoke('Who is president of india')
print(response.content)

c:\Python\Python311\Lib\site-packages\pydantic\json_schema.py:2279: PydanticJsonSchemaWarning: Default value typing.Optional[FieldInfo(annotation=NoneType, required=True, description='person or system that interacts with the product or feature described in the story')] is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Python\Python311\Lib\site-packages\pydantic\json_schema.py:2279: PydanticJsonSchemaWarning: Default value typing.Optional[FieldInfo(annotation=NoneType, required=True, description='a subsection or part of a project that helps you organize and group')] is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


AttributeError: 'NoneType' object has no attribute 'beta'

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict
from langchain_core.prompts import PromptTemplate

load_dotenv()

model = ChatOpenAI()

# schema
class JiraEpic(BaseModel): 
    actors: str = Optional[Field(description='person or system that interacts with the product or feature described in the story')]
    Components: List[str] = Optional[Field(description='a subsection or part of a project that helps you organize and group')]

structured_model = model.with_structured_output(JiraEpic)

# template
template = PromptTemplate(
    template="""
You are a technical architect and Jira expert.
Find the Actors and Components in the Jira Epic Story described below:
"{jira_epic}"
""",
input_variables=['jira_epic']
)
prompt = template.invoke({'jira_epic':epic})
result = structured_model.invoke(prompt)

print(result)

2.10.6


In [ ]:
import random, subprocess, tempfile, os
from langchain_core.tools import tool

def render_mermaid(code: str, fmt: str = "png") -> bytes:
    """
    Render Mermaid code to PNG or SVG using mermaid-cli (mmdc).
    Requires: npm install -g @mermaid-js/mermaid-cli
    """
    # Ensure format is supported
    fmt = fmt.lower()
    if fmt not in {"png", "svg"}:
        raise ValueError("Format must be 'png' or 'svg'")

    with tempfile.TemporaryDirectory() as tmpdir:
        input_file = os.path.join(tmpdir, "diagram.mmd")
        output_file = os.path.join(tmpdir, f"diagram.{fmt}")

        # Write mermaid code to file
        with open(input_file, "w", encoding="utf-8") as f:
            f.write(code)

        #command = f"mmdc -i {input_file} -o {output_file}"
        # Call mmdc (Mermaid CLI)
        subprocess.run(
            ["mmdc", "-i", input_file, "-o", output_file],    
            check=False,
            shell=True
        )

        # Read the rendered output
        with open(output_file, "rb") as f:
            return f.read()

In [20]:
render_mermaid("flowchart TD; A-->B; B-->C;")